# Monte Carlo validation of CRYSTRACT uncertainty propagation

Supplementary notebook for manuscript lan5011, Supporting Information Section S2.3.

Run with a Python 3 Jupyter kernel. The calculation calls `Rscript`, with `crystract` (corrected source), `data.table` and their dependencies installed in R. This notebook embeds the companion R validation function so it does not depend on a local repository path.

The synthetic example samples 12 independent original inputs, regenerates symmetry-related neighbours from the same sampled parent, and compares 4,000 simulated distances and angles with analytic standard uncertainties. Seed: 10. This checks the stated non-collinear triclinic case; it does not recover missing refinement covariance or validate all case-study CIFs.

Cell angles and their standard uncertainties are supplied in degrees, converted to radians for sampling and differentiation, and reported in degrees. Lengths use angstroms; coordinates are fractional.

Method: JCGM 100:2008 and JCGM 101:2008. Companion implementation: [validate_uncertainty_monte_carlo.R](https://github.com/PrabhuLab/crystract-in-action/blob/48b43aee3bfc839db4c225b84d38f0e6e799e026/scripts/validate_uncertainty_monte_carlo.R).


In [1]:
import subprocess

r_code = 'run_monte_carlo_validation <- function(samples = 4000L, seed = 10L) {\n  stopifnot(length(samples) == 1L, is.finite(samples), samples >= 2, samples == as.integer(samples))\n  cell <- data.table::data.table(\n    `_cell_length_a` = 7, `_cell_length_b` = 8, `_cell_length_c` = 9,\n    `_cell_angle_alpha` = 79, `_cell_angle_beta` = 83, `_cell_angle_gamma` = 74,\n    `_cell_length_a_error` = .002, `_cell_length_b_error` = .002, `_cell_length_c_error` = .002,\n    `_cell_angle_alpha_error` = .01, `_cell_angle_beta_error` = .01, `_cell_angle_gamma_error` = .01)\n  atoms <- data.table::data.table(Label = c("A", "B"), x_a = c(.1,.2), y_b = c(.05,.12),\n    z_c = c(.07,.3), x_error = .001, y_error = .001, z_error = .001)\n  operations <- data.table::data.table(x = c("x","-x"), y = c("y","-y"), z = c("z","-z"))\n  images <- crystract::expand_transformed_coords(crystract::apply_symmetry_operations(atoms, operations, cell), c(0,0,0))\n  bond <- crystract::calculate_distances(atoms[Label == "A"], images[Label == "B_2_0_0_0"], cell)\n  angle <- data.table::data.table(CentralAtom = "A", Neighbor1 = "B_1_0_0_0", Neighbor2 = "B_2_0_0_0", Angle = 0)\n  analytic <- c(crystract::propagate_distance_error(bond, atoms, cell, images)$DistanceError,\n                crystract::propagate_angle_error(angle, atoms, images, cell)$AngleError)\n  mean_input <- c(7,8,9,c(79,83,74)*pi/180,.1,.05,.07,.2,.12,.3)\n  sd_input <- c(rep(.002,3),rep(.01*pi/180,3),rep(.001,6))\n  evaluate <- function(p) { # p = (a,b,c,alpha,beta,gamma,A_x,A_y,A_z,B_x,B_y,B_z); angles in radians.\n    ca <- cos(p[4]); cb <- cos(p[5]); cg <- cos(p[6]); sg <- sin(p[6])\n    D <- 1-ca^2-cb^2-cg^2+2*ca*cb*cg\n    M <- rbind(c(p[1],p[2]*cg,p[3]*cb), c(0,p[2]*sg,p[3]*(ca-cb*cg)/sg), c(0,0,p[3]*sqrt(D)/sg))\n    first <- as.numeric(M %*% (p[10:12]-p[7:9])) # u = M(B-A).\n    second <- as.numeric(M %*% (1-p[10:12]-p[7:9])) # v = M((1-B)-A); reuse the same sampled B.\n    c(Distance = sqrt(sum(second^2)), Angle = acos(sum(first*second)/sqrt(sum(first^2)*sum(second^2)))*180/pi)\n  }\n  set.seed(seed)\n  inputs <- matrix(stats::rnorm(samples*12,mean_input,sd_input),ncol=12,byrow=TRUE)\n  simulated <- t(apply(inputs,1,evaluate))\n  observed <- apply(simulated,2,stats::sd) # Sample SD of q(p) estimates sigma_q without analytic derivatives.\n  summary <- data.table::data.table(Quantity=c("Distance","Angle"),Unit=c("angstrom","degree"),\n    AnalyticSU=analytic, MonteCarloSD=unname(observed),\n    RelativeDifferencePercent=100*abs(analytic/observed-1), Samples=samples, Seed=seed)\n  stopifnot(all(is.finite(simulated)),all(is.finite(analytic)),all(analytic > 0))\n  input_table <- data.table::data.table(Parameter=c("a","b","c","alpha","beta","gamma","A_x","A_y","A_z","B_x","B_y","B_z"),\n    Mean=c(7,8,9,79,83,74,.1,.05,.07,.2,.12,.3), StandardUncertainty=c(rep(.002,3),rep(.01,3),rep(.001,6)),\n    Unit=c(rep("angstrom",3),rep("degree",3),rep("fractional",6)))\n  list(summary=summary, inputs=input_table, draws=data.table::as.data.table(simulated), nominal=evaluate(mean_input))\n}\n\nexport_monte_carlo_validation <- function(result, output_dir) {\n  dir.create(output_dir,recursive=TRUE,showWarnings=FALSE)\n  data.table::fwrite(result$summary,file.path(output_dir,"monte_carlo_summary.csv"))\n  data.table::fwrite(result$inputs,file.path(output_dir,"monte_carlo_inputs.csv"))\n  data.table::fwrite(result$draws,file.path(output_dir,"monte_carlo_draws.csv"))\n  long <- data.table::melt(result$draws,measure.vars=c("Distance","Angle"),variable.name="Quantity",value.name="Value")\n  curve <- data.table::rbindlist(lapply(seq_len(nrow(result$summary)),function(i) {\n    q <- result$summary$Quantity[i]\n    x <- seq(min(long[Quantity==q,Value]),max(long[Quantity==q,Value]),length.out=400)\n    data.table::data.table(Quantity=q,Value=x,Density=stats::dnorm(x,result$nominal[q],result$summary$AnalyticSU[i]))\n  }))\n  plot <- ggplot2::ggplot(long,ggplot2::aes(x=Value)) +\n    ggplot2::geom_histogram(ggplot2::aes(y=ggplot2::after_stat(density)),bins=45,fill="#74a9cf",color="white") +\n    ggplot2::geom_line(data=curve,ggplot2::aes(y=Density),color="#a50f15",linewidth=.9) +\n    ggplot2::facet_wrap(~Quantity,scales="free",labeller=ggplot2::as_labeller(c(Distance="Distance (angstrom)",Angle="Angle (degrees)"))) +\n    ggplot2::labs(x="Recalculated value",y="Probability density",title="Monte Carlo validation of propagated uncertainty",\n      subtitle=sprintf("%s draws; bars: simulated values; line: normal density using the analytic standard uncertainty",result$summary$Samples[1])) +\n    ggplot2::theme_bw(base_size=14)\n  ggplot2::ggsave(file.path(output_dir,"monte_carlo_validation.png"),plot,width=11,height=4.5,dpi=400)\n  invisible(plot)\n}\n\nresult <- run_monte_carlo_validation(samples=4000L, seed=10L)\nstopifnot(all(result$summary$RelativeDifferencePercent < 6))\nprint(result$inputs)\nprint(result$summary)\ncat("\\nCell-angle inputs in this table are degrees; sampling and derivatives use radians. Output angles use degrees.\\n")\n'

result = subprocess.run(["Rscript", "-e", r_code], text=True, capture_output=True, check=True)
print(result.stdout)


    Parameter  Mean StandardUncertainty       Unit
       <char> <num>               <num>     <char>
 1:         a  7.00               0.002   angstrom
 2:         b  8.00               0.002   angstrom
 3:         c  9.00               0.002   angstrom
 4:     alpha 79.00               0.010     degree
 5:      beta 83.00               0.010     degree
 6:     gamma 74.00               0.010     degree
 7:       A_x  0.10               0.001 fractional
 8:       A_y  0.05               0.001 fractional
 9:       A_z  0.07               0.001 fractional
10:       B_x  0.20               0.001 fractional
11:       B_y  0.12               0.001 fractional
12:       B_z  0.30               0.001 fractional
   Quantity     Unit AnalyticSU MonteCarloSD RelativeDifferencePercent Samples
     <char>   <char>      <num>        <num>                     <num>   <int>
1: Distance angstrom 0.01356797   0.01352752                 0.2989735    4000
2:    Angle   degree 0.26437895   0.26542867     